In [1]:
pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.7 MB/s eta 0:00:00


In [5]:
pip install openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 13.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 1.7 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=3798510872dcd89206597bdc021f8d75b145d9b7ca6071adb667b223398ebbf7
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


In [7]:
import whisper
from groq import Groq
import json

# -----------------------------
# Step 1: Convert Audio to Text
# -----------------------------

model = whisper.load_model("base")

result = model.transcribe("/content/team_meeting.wav")
transcribed_text = result["text"]

print("Transcribed Text:")
print(transcribed_text)

# -----------------------------
# Step 2: Groq API
# -----------------------------

client = Groq(
    api_key="YOUR_API_KEY"
)

prompt = """
Extract the following meeting information from the text.

Return ONLY valid JSON.

Fields:
- meeting
- date
- time
- person
- location

Text:
"{}"
""".format(transcribed_text)

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0
)

output = response.choices[0].message.content

# Clean the output string to remove markdown code block delimiters
if output.startswith('```json') and output.endswith('```'):
    output = output[len('```json'):-len('```')].strip()

# -----------------------------
# Step 3: Print JSON Output
# -----------------------------

meeting_details = json.loads(output)

print("\nMeeting Details:")
print(json.dumps(meeting_details, indent=4))


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Transcribed Text:
 Our project review meeting is on Friday at 3pm with Mr. Rajesh in conference room B.

Meeting Details:
{
    "meeting": "project review meeting",
    "date": "Friday",
    "time": "3pm",
    "person": "Mr. Rajesh",
    "location": "conference room B"
}
